In [1]:
import matplotlib.pyplot as plt
import numpy as np
import joblib
import json
import pandas as pd

In [2]:
final_model = joblib.load(
    "../models/churn_logistic_model.joblib"
)

with open("../models/model_metadata.json", "r") as f:
    metadata = json.load(f)

threshold = metadata["threshold"]

print(f"Threshold: {threshold}")

Threshold: 0.3


In [3]:
df = pd.read_csv("../data/raw/data.csv")

In [4]:
preprocessor = final_model.named_steps["preprocessor"]
model = final_model.named_steps["model"]

feature_names = preprocessor.get_feature_names_out()

print("Number of transformed features:", len(feature_names))
print(feature_names)

Number of transformed features: 46
['categorical__gender_Female' 'categorical__gender_Male'
 'categorical__SeniorCitizen_0' 'categorical__SeniorCitizen_1'
 'categorical__Partner_No' 'categorical__Partner_Yes'
 'categorical__Dependents_No' 'categorical__Dependents_Yes'
 'categorical__PhoneService_No' 'categorical__PhoneService_Yes'
 'categorical__MultipleLines_No'
 'categorical__MultipleLines_No phone service'
 'categorical__MultipleLines_Yes' 'categorical__InternetService_DSL'
 'categorical__InternetService_Fiber optic'
 'categorical__InternetService_No' 'categorical__OnlineSecurity_No'
 'categorical__OnlineSecurity_No internet service'
 'categorical__OnlineSecurity_Yes' 'categorical__OnlineBackup_No'
 'categorical__OnlineBackup_No internet service'
 'categorical__OnlineBackup_Yes' 'categorical__DeviceProtection_No'
 'categorical__DeviceProtection_No internet service'
 'categorical__DeviceProtection_Yes' 'categorical__TechSupport_No'
 'categorical__TechSupport_No internet service'
 'ca

In [5]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

df["TotalCharges"] = df["TotalCharges"].fillna(0)

In [6]:
customer_original = df.iloc[[0]].copy()

customer_original

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No


In [7]:
actual_churn = customer_original["Churn"].iloc[0]
customer_id = customer_original["customerID"].iloc[0]

customer = customer_original.drop(
    columns=["customerID", "Churn"]
)



In [8]:
churn_probability = final_model.predict_proba(
    customer
)[0, 1]

prediction = int(
    churn_probability >= threshold
)

print("Customer:", customer_id)
print("Churn probability:", churn_probability)
print("Threshold:", threshold)
print("Prediction:", prediction)
print("Actual churn:", actual_churn)

Customer: 7590-VHVEG
Churn probability: 0.6413025841448515
Threshold: 0.3
Prediction: 1
Actual churn: No


In [9]:
transformed_customer = preprocessor.transform(
    customer
)

In [10]:
if hasattr(transformed_customer, "toarray"):
    transformed_customer = transformed_customer.toarray()

In [11]:
print(transformed_customer.shape)

(1, 46)


In [12]:
coefficients = model.coef_[0]

In [13]:
customer_values = transformed_customer[0]

contributions = (
    customer_values * coefficients
)

In [14]:
contribution_df = pd.DataFrame({
    "feature": feature_names,
    "customer_value": customer_values,
    "coefficient": coefficients,
    "contribution": contributions
})

In [15]:
contribution_df = contribution_df.sort_values(
    "contribution",
    ascending=False
)

contribution_df.head(10)

,feature,customer_value,coefficient,contribution
43,numerical__tenure,-1.277173,-1.310581,1.673839
34,categorical__Contract_Month-to-month,1.000000,0.633380,0.633380
44,numerical__MonthlyCharges,-1.160489,-0.396531,0.460170
41,categorical__PaymentMethod_Electronic check,1.000000,0.237171,0.237171
25,categorical__TechSupport_No,1.000000,0.154527,0.154527
16,categorical__OnlineSecurity_No,1.000000,0.149349,0.149349
38,categorical__PaperlessBilling_Yes,1.000000,0.082104,0.082104
10,categorical__MultipleLines_No,0.000000,-0.275739,-0.000000
1,categorical__gender_Male,0.000000,-0.125772,-0.000000
3,categorical__SeniorCitizen_1,0.000000,-0.042104,-0.000000


In [16]:
intercept = model.intercept_[0]

In [17]:
logit = (
    intercept +
    contributions.sum()
)

print(logit)

0.5810222153710297


In [18]:
probability_manual = 1 / (
    1 + np.exp(-logit)
)

print("Manual probability:", probability_manual)
print("Model probability:", churn_probability)

Manual probability: 0.6413025841448515
Model probability: 0.6413025841448515


In [19]:
risk_drivers = (
    contribution_df[
        contribution_df["contribution"] > 0
    ]
    .sort_values(
        "contribution",
        ascending=False
    )
    .head(5)
)

risk_drivers

,feature,customer_value,coefficient,contribution
43,numerical__tenure,-1.277173,-1.310581,1.673839
34,categorical__Contract_Month-to-month,1.000000,0.633380,0.633380
44,numerical__MonthlyCharges,-1.160489,-0.396531,0.460170
41,categorical__PaymentMethod_Electronic check,1.000000,0.237171,0.237171
25,categorical__TechSupport_No,1.000000,0.154527,0.154527


In [20]:
protective_factors = (
    contribution_df[
        contribution_df["contribution"] < 0
    ]
    .sort_values(
        "contribution",
        ascending=True
    )
    .head(5)
)

protective_factors

,feature,customer_value,coefficient,contribution
45,numerical__TotalCharges,-0.99169,0.600618,-0.595627
13,categorical__InternetService_DSL,1.00000,-0.558971,-0.558971
2,categorical__SeniorCitizen_0,1.00000,-0.230216,-0.230216
31,categorical__StreamingMovies_No,1.00000,-0.163195,-0.163195
28,categorical__StreamingTV_No,1.00000,-0.162012,-0.162012


In [21]:
categorical_columns = list(
    preprocessor.transformers_[0][2]
)

In [22]:
def readable_feature_name(feature_name):

    # Numerical feature
    if feature_name.startswith("numerical__"):
        return feature_name.replace(
            "numerical__",
            ""
        )

    # Categorical feature
    if feature_name.startswith("categorical__"):

        cleaned = feature_name.replace(
            "categorical__",
            ""
        )

        for column in categorical_columns:

            prefix = column + "_"

            if cleaned.startswith(prefix):

                category = cleaned[len(prefix):]

                return f"{column} = {category}"

    return feature_name

In [23]:
contribution_df["readable_feature"] = (
    contribution_df["feature"]
    .apply(readable_feature_name)
)

In [24]:
risk_drivers = (
    contribution_df[
        contribution_df["contribution"] > 0
    ]
    .sort_values(
        "contribution",
        ascending=False
    )
    .head(5)
)

protective_factors = (
    contribution_df[
        contribution_df["contribution"] < 0
    ]
    .sort_values(
        "contribution",
        ascending=True
    )
    .head(5)
)

In [25]:
risk_drivers[
    ["readable_feature", "contribution"]
]

,readable_feature,contribution
43,tenure,1.673839
34,Contract = Month-to-month,0.633380
44,MonthlyCharges,0.460170
41,PaymentMethod = Electronic check,0.237171
25,TechSupport = No,0.154527


In [26]:
protective_factors[
    ["readable_feature", "contribution"]
]

,readable_feature,contribution
45,TotalCharges,-0.595627
13,InternetService = DSL,-0.558971
2,SeniorCitizen = 0,-0.230216
31,StreamingMovies = No,-0.163195
28,StreamingTV = No,-0.162012


In [27]:
numerical_columns = list(
    preprocessor.transformers_[1][2]
)

scaler = preprocessor.named_transformers_["numerical"]

print(numerical_columns)
print(scaler.mean_)

['tenure', 'MonthlyCharges', 'TotalCharges']
[  32.34682286   64.81390664 2283.56118211]


In [28]:
training_means = dict(
    zip(numerical_columns, scaler.mean_)
)

training_means

{'tenure': np.float64(32.34682286119986),
 'MonthlyCharges': np.float64(64.81390663826767),
 'TotalCharges': np.float64(2283.561182108626)}

In [29]:
def describe_numerical_feature(feature, customer):

    value = float(customer[feature].iloc[0])
    mean = training_means[feature]

    if value < mean:
        comparison = "below average"
    else:
        comparison = "above average"

    if feature == "tenure":
        return (
            f"Tenure: {value:.0f} months "
            f"({comparison})"
        )

    if feature == "MonthlyCharges":
        return (
            f"Monthly charges: ${value:.2f} "
            f"({comparison})"
        )

    if feature == "TotalCharges":
        return (
            f"Total charges: ${value:.2f} "
            f"({comparison})"
        )

    return f"{feature}: {value:.2f} ({comparison})"

In [30]:
describe_numerical_feature(
    "tenure",
    customer
)

'Tenure: 1 months (below average)'

In [31]:
def create_explanation(row):

    feature = row["readable_feature"]

    if feature in numerical_columns:
        return describe_numerical_feature(
            feature,
            customer
        )

    return feature

In [32]:
contribution_df["explanation"] = (
    contribution_df.apply(
        create_explanation,
        axis=1
    )
)

In [33]:
risk_drivers = (
    contribution_df[
        contribution_df["contribution"] > 0
    ]
    .sort_values(
        "contribution",
        ascending=False
    )
    .head(5)
)

protective_factors = (
    contribution_df[
        contribution_df["contribution"] < 0
    ]
    .sort_values(
        "contribution",
        ascending=True
    )
    .head(5)
)

In [34]:
risk_drivers[
    ["explanation", "contribution"]
]
protective_factors[
    ["explanation", "contribution"]
]

,explanation,contribution
45,Total charges: $29.85 (below average),-0.595627
13,InternetService = DSL,-0.558971
2,SeniorCitizen = 0,-0.230216
31,StreamingMovies = No,-0.163195
28,StreamingTV = No,-0.162012


In [35]:
def get_base_feature(feature_name):

    if feature_name.startswith("numerical__"):
        return feature_name.replace(
            "numerical__",
            ""
        )

    if feature_name.startswith("categorical__"):

        cleaned = feature_name.replace(
            "categorical__",
            ""
        )

        for column in categorical_columns:

            prefix = column + "_"

            if cleaned.startswith(prefix):
                return column

    return feature_name

In [36]:
contribution_df["base_feature"] = (
    contribution_df["feature"]
    .apply(get_base_feature)
)

In [37]:
contribution_df[
    ["feature", "base_feature", "contribution"]
].head()

,feature,base_feature,contribution
43,numerical__tenure,tenure,1.673839
34,categorical__Contract_Month-to-month,Contract,0.633380
44,numerical__MonthlyCharges,MonthlyCharges,0.460170
41,categorical__PaymentMethod_Electronic check,PaymentMethod,0.237171
25,categorical__TechSupport_No,TechSupport,0.154527


In [38]:
actionable_features = {
    "Contract",
    "TechSupport",
    "PaymentMethod",
    "MonthlyCharges",
    "tenure",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection"
}

In [39]:
contribution_df["actionable"] = (
    contribution_df["base_feature"]
    .isin(actionable_features)
)

In [40]:
contribution_df[
    ["explanation", "contribution", "actionable"]
]

,explanation,contribution,actionable
43,Tenure: 1 months (below average),1.673839,True
34,Contract = Month-to-month,0.633380,True
44,Monthly charges: $29.85 (below average),0.460170,True
41,PaymentMethod = Electronic check,0.237171,True
25,TechSupport = No,0.154527,True
16,OnlineSecurity = No,0.149349,True
38,PaperlessBilling = Yes,0.082104,False
10,MultipleLines = No,-0.000000,False
1,gender = Male,-0.000000,False
3,SeniorCitizen = 1,-0.000000,False


In [41]:
actionable_risk_drivers = (
    contribution_df[
        (contribution_df["contribution"] > 0)
        &
        (contribution_df["actionable"])
    ]
    .sort_values(
        "contribution",
        ascending=False
    ).head(5)
)

In [42]:
actionable_risk_drivers[
    [
        "explanation",
        "contribution",
        "base_feature"
    ]
].head(5)

,explanation,contribution,base_feature
43,Tenure: 1 months (below average),1.673839,tenure
34,Contract = Month-to-month,0.633380,Contract
44,Monthly charges: $29.85 (below average),0.460170,MonthlyCharges
41,PaymentMethod = Electronic check,0.237171,PaymentMethod
25,TechSupport = No,0.154527,TechSupport


In [43]:
def get_retention_action(feature, customer):

    if feature == "Contract":

        if customer["Contract"].iloc[0] == "Month-to-month":
            return "Offer an incentive to move to a longer-term contract"

    if feature == "TechSupport":

        if customer["TechSupport"].iloc[0] == "No":
            return "Offer a free or discounted tech-support trial"

    if feature == "PaymentMethod":

        if customer["PaymentMethod"].iloc[0] == "Electronic check":
            return "Offer an incentive to switch to automatic payment"

    if feature == "tenure":

        tenure = customer["tenure"].iloc[0]

        if tenure < 12:
            return "Place customer in an early-life retention or onboarding campaign"

    if feature == "MonthlyCharges":
        monthly_charge = customer[
        "MonthlyCharges" ].iloc[0]
        if monthly_charge > training_means["MonthlyCharges"]:
            return ("Review the customer's current plan, pricing, and bundle")

    if feature == "OnlineSecurity":

        if customer["OnlineSecurity"].iloc[0] == "No":
            return "Consider an Online Security trial or bundled offer"

    if feature == "OnlineBackup":

        if customer["OnlineBackup"].iloc[0] == "No":
            return "Consider an Online Backup trial or bundled offer"

    if feature == "DeviceProtection":

        if customer["DeviceProtection"].iloc[0] == "No":
            return "Consider a Device Protection offer"

    return None

In [44]:
recommendations = []

for _, row in actionable_risk_drivers.iterrows():

    action = get_retention_action(
        row["base_feature"],
        customer
    )

    if action is not None:

        recommendations.append({
            "driver": row["explanation"],
            "contribution": row["contribution"],
            "recommended_action": action
        })

In [45]:
print("Customer:", customer_id)
print("Churn probability:", churn_probability)

print("\nTop risk drivers:")
display(
    risk_drivers[
        ["explanation", "contribution"]
    ]
)

print("\nSuggested actions:")
for recommendation in recommendations:
    print("-", recommendation["recommended_action"])

Customer: 7590-VHVEG
Churn probability: 0.6413025841448515

Top risk drivers:


,explanation,contribution
43,Tenure: 1 months (below average),1.673839
34,Contract = Month-to-month,0.633380
44,Monthly charges: $29.85 (below average),0.460170
41,PaymentMethod = Electronic check,0.237171
25,TechSupport = No,0.154527



Suggested actions:
- Place customer in an early-life retention or onboarding campaign
- Offer an incentive to move to a longer-term contract
- Offer an incentive to switch to automatic payment
- Offer a free or discounted tech-support trial


In [46]:
def analyze_customer(customer, final_model, threshold, top_n=5):
    """
    Analyze one telecom customer.

    Returns:
    - churn probability
    - churn prediction
    - top risk drivers
    - top protective factors
    - suggested retention actions
    """

    # ---------------------------------------------------------
    # 1. Make sure customer is a one-row DataFrame
    # ---------------------------------------------------------

    if isinstance(customer, pd.Series):
        customer = customer.to_frame().T

    customer = customer.copy()

    if len(customer) != 1:
        raise ValueError("analyze_customer() expects exactly one customer.")


    # ---------------------------------------------------------
    # 2. Save customer ID if available
    # ---------------------------------------------------------

    customer_id = None

    if "customerID" in customer.columns:
        customer_id = customer["customerID"].iloc[0]


    # ---------------------------------------------------------
    # 3. Basic cleaning
    # ---------------------------------------------------------

    if "TotalCharges" in customer.columns:
        customer["TotalCharges"] = pd.to_numeric(
            customer["TotalCharges"],
            errors="coerce"
        ).fillna(0)


    # ---------------------------------------------------------
    # 4. Extract model components
    # ---------------------------------------------------------

    preprocessor = final_model.named_steps["preprocessor"]
    model = final_model.named_steps["model"]

    expected_features = list(
        final_model.feature_names_in_
    )

    model_input = customer[expected_features]


    # ---------------------------------------------------------
    # 5. Predict churn probability
    # ---------------------------------------------------------

    churn_probability = final_model.predict_proba(
        model_input
    )[0, 1]

    prediction = int(
        churn_probability >= threshold
    )


    # ---------------------------------------------------------
    # 6. Get transformed feature names
    # ---------------------------------------------------------

    feature_names = (
        preprocessor.get_feature_names_out()
    )

    categorical_columns = list(
        preprocessor.transformers_[0][2]
    )

    numerical_columns = list(
        preprocessor.transformers_[1][2]
    )


    # ---------------------------------------------------------
    # 7. Transform this customer exactly as the model does
    # ---------------------------------------------------------

    transformed_customer = preprocessor.transform(
        model_input
    )

    if hasattr(transformed_customer, "toarray"):
        transformed_customer = transformed_customer.toarray()

    customer_values = transformed_customer[0]


    # ---------------------------------------------------------
    # 8. Calculate individual feature contributions
    # ---------------------------------------------------------

    coefficients = model.coef_[0]

    contributions = (
        customer_values * coefficients
    )


    # ---------------------------------------------------------
    # 9. Get training averages from fitted scaler
    # ---------------------------------------------------------

    scaler = (
        preprocessor
        .named_transformers_["numerical"]
    )

    training_means = dict(
        zip(
            numerical_columns,
            scaler.mean_
        )
    )


    # ---------------------------------------------------------
    # 10. Helper: identify original feature
    # ---------------------------------------------------------

    def get_base_feature(feature_name):

        if feature_name.startswith("numerical__"):
            return feature_name.replace(
                "numerical__",
                ""
            )

        if feature_name.startswith("categorical__"):

            cleaned = feature_name.replace(
                "categorical__",
                ""
            )

            for column in categorical_columns:

                prefix = column + "_"

                if cleaned.startswith(prefix):
                    return column

        return feature_name


    # ---------------------------------------------------------
    # 11. Helper: create readable explanation
    # ---------------------------------------------------------

    def create_explanation(feature_name):

        base_feature = get_base_feature(
            feature_name
        )

        # Numerical features
        if base_feature in numerical_columns:

            value = float(
                model_input[
                    base_feature
                ].iloc[0]
            )

            mean = training_means[
                base_feature
            ]

            comparison = (
                "below average"
                if value < mean
                else "above average"
            )

            if base_feature == "tenure":
                return (
                    f"Tenure: {value:.0f} months "
                    f"({comparison})"
                )

            if base_feature == "MonthlyCharges":
                return (
                    f"Monthly charges: "
                    f"${value:.2f} "
                    f"({comparison})"
                )

            if base_feature == "TotalCharges":
                return (
                    f"Total charges: "
                    f"${value:.2f} "
                    f"({comparison})"
                )

            return (
                f"{base_feature}: "
                f"{value:.2f} "
                f"({comparison})"
            )


        # Categorical features
        cleaned = feature_name.replace(
            "categorical__",
            ""
        )

        for column in categorical_columns:

            prefix = column + "_"

            if cleaned.startswith(prefix):

                category = cleaned[
                    len(prefix):
                ]

                return (
                    f"{column} = {category}"
                )

        return cleaned


    # ---------------------------------------------------------
    # 12. Build contribution table
    # ---------------------------------------------------------

    contribution_df = pd.DataFrame({
        "feature": feature_names,
        "customer_value": customer_values,
        "coefficient": coefficients,
        "contribution": contributions
    })

    contribution_df["base_feature"] = (
        contribution_df["feature"]
        .apply(get_base_feature)
    )

    contribution_df["explanation"] = (
        contribution_df["feature"]
        .apply(create_explanation)
    )


    # ---------------------------------------------------------
    # 13. Get strongest risk drivers
    # ---------------------------------------------------------

    risk_drivers = (
        contribution_df[
            contribution_df["contribution"] > 0
        ]
        .sort_values(
            "contribution",
            ascending=False
        )
        .head(top_n)
    )


    # ---------------------------------------------------------
    # 14. Get strongest protective factors
    # ---------------------------------------------------------

    protective_factors = (
        contribution_df[
            contribution_df["contribution"] < 0
        ]
        .sort_values(
            "contribution",
            ascending=True
        )
        .head(top_n)
    )


    # ---------------------------------------------------------
    # 15. Retention recommendation rules
    # ---------------------------------------------------------

    def get_retention_action(feature):

        if feature == "Contract":

            if (
                model_input["Contract"].iloc[0]
                == "Month-to-month"
            ):
                return (
                    "Offer an incentive to move "
                    "to a longer-term contract"
                )


        if feature == "TechSupport":

            if (
                model_input["TechSupport"].iloc[0]
                == "No"
            ):
                return (
                    "Offer a free or discounted "
                    "tech-support trial"
                )


        if feature == "PaymentMethod":

            if (
                model_input[
                    "PaymentMethod"
                ].iloc[0]
                == "Electronic check"
            ):
                return (
                    "Offer an incentive to switch "
                    "to automatic payment"
                )


        if feature == "tenure":

            tenure = model_input[
                "tenure"
            ].iloc[0]

            if tenure < 12:
                return (
                    "Place customer in an early-life "
                    "retention or onboarding campaign"
                )


        if feature == "MonthlyCharges":

            monthly_charge = model_input[
                "MonthlyCharges"
            ].iloc[0]

            if (
                monthly_charge >
                training_means["MonthlyCharges"]
            ):
                return (
                    "Review the customer's current "
                    "plan, pricing, and bundle"
                )


        if feature == "OnlineSecurity":

            if (
                model_input[
                    "OnlineSecurity"
                ].iloc[0]
                == "No"
            ):
                return (
                    "Consider an Online Security "
                    "trial or bundled offer"
                )


        if feature == "OnlineBackup":

            if (
                model_input[
                    "OnlineBackup"
                ].iloc[0]
                == "No"
            ):
                return (
                    "Consider an Online Backup "
                    "trial or bundled offer"
                )


        if feature == "DeviceProtection":

            if (
                model_input[
                    "DeviceProtection"
                ].iloc[0]
                == "No"
            ):
                return (
                    "Consider a Device Protection "
                    "trial or bundled offer"
                )

        return None


    # ---------------------------------------------------------
    # 16. Only generate actions from displayed risk drivers
    # ---------------------------------------------------------

    recommendations = []

    for _, row in risk_drivers.iterrows():

        action = get_retention_action(
            row["base_feature"]
        )

        if action is not None:

            recommendations.append({
                "driver": row["explanation"],
                "action": action
            })


    # ---------------------------------------------------------
    # 17. Clean output into normal Python structures
    # ---------------------------------------------------------

    risk_output = []

    for _, row in risk_drivers.iterrows():

        risk_output.append({
            "feature": row["base_feature"],
            "explanation": row["explanation"],
            "contribution": float(
                row["contribution"]
            )
        })


    protective_output = []

    for _, row in protective_factors.iterrows():

        protective_output.append({
            "feature": row["base_feature"],
            "explanation": row["explanation"],
            "contribution": float(
                row["contribution"]
            )
        })


    # ---------------------------------------------------------
    # 18. Return complete customer analysis
    # ---------------------------------------------------------

    return {
        "customer_id": customer_id,
        "churn_probability": float(
            churn_probability
        ),
        "threshold": float(threshold),
        "prediction": prediction,
        "prediction_label": (
            "Likely to Churn"
            if prediction == 1
            else "Likely to Stay"
        ),
        "risk_drivers": risk_output,
        "protective_factors": protective_output,
        "suggested_actions": recommendations
    }

In [47]:
analysis = analyze_customer(
    customer,
    final_model,
    threshold,
    top_n=5
)

In [48]:
analysis

{'customer_id': None,
 'churn_probability': 0.6413025841448515,
 'threshold': 0.3,
 'prediction': 1,
 'prediction_label': 'Likely to Churn',
 'risk_drivers': [{'feature': 'tenure',
   'explanation': 'Tenure: 1 months (below average)',
   'contribution': 1.6738388299897},
  {'feature': 'Contract',
   'explanation': 'Contract = Month-to-month',
   'contribution': 0.6333795039616322},
  {'feature': 'MonthlyCharges',
   'explanation': 'Monthly charges: $29.85 (below average)',
   'contribution': 0.4601702832723339},
  {'feature': 'PaymentMethod',
   'explanation': 'PaymentMethod = Electronic check',
   'contribution': 0.2371714402588073},
  {'feature': 'TechSupport',
   'explanation': 'TechSupport = No',
   'contribution': 0.15452655905152574}],
 'protective_factors': [{'feature': 'TotalCharges',
   'explanation': 'Total charges: $29.85 (below average)',
   'contribution': -0.5956270368135733},
  {'feature': 'InternetService',
   'explanation': 'InternetService = DSL',
   'contribution': -